In [53]:
import pandas as pd
import numpy as np

# 1. Read the file, skipping the first 78 rows of metadata
file_path = "GSE42568_series_matrix.txt"
df = pd.read_csv(file_path, sep='\t', skiprows=78)

# Set the very first column as 'Gene_ID'
df.rename(columns={df.columns[0]: 'Gene_ID'}, inplace=True)

# Select exactly the first 40 rows (genes)
df_selected = df.head(40).copy()

# 2. Define the columns for Normal and Cancer samples
normal_cols = df_selected.columns[1:18]
cancer_cols = df_selected.columns[18:]

# 3. Calculate Averages, Fold Change, and Direction
df_selected['Average expression in normal samples'] = df_selected[normal_cols].mean(axis=1)
df_selected['Average expression in breast cancer samples'] = df_selected[cancer_cols].mean(axis=1)
df_selected['Fold change'] = df_selected['Average expression in breast cancer samples'] - df_selected['Average expression in normal samples']
df_selected['Direction of change: increased or decreased'] = np.where(df_selected['Fold change'] > 0, 'increased', 'decreased')

# 4. Dictionary mapping Probe IDs to both Official Gene Symbols AND Biological Notes separately
gene_mapping = {
    "1007_s_at": {"symbol": "DDR1", "note": "Epithelial cell adhesion and receptor tyrosine kinase signaling."},
    "1053_at": {"symbol": "RFC2", "note": "DNA replication elongation and chromosomal stability."},
    "117_at": {"symbol": "HSPA6", "note": "Heat-shock protein active during cellular stress and apoptosis."},
    "121_at": {"symbol": "PAX8", "note": "Transcription factor involved in developmental regulation."},
    "1255_g_at": {"symbol": "GUCA1A", "note": "Guanylyl cyclase activator in calcium signaling pathways."},
    "1294_at": {"symbol": "UBA7", "note": "Ubiquitin activating enzyme involved in immune response."},
    "1316_at": {"symbol": "THRA", "note": "Thyroid hormone receptor alpha, regulates gene expression."},
    "1320_at": {"symbol": "PTPN21", "note": "Protein tyrosine phosphatase, regulates cell growth and differentiation."},
    "1405_i_at": {"symbol": "CCL5", "note": "Chemokine involved in immunoregulatory and inflammatory processes."},
    "1431_at": {"symbol": "CYP2E1", "note": "Cytochrome P450 enzyme, metabolizes carcinogens and toxins."},
    "1438_at": {"symbol": "EPHB3", "note": "Ephrin receptor, involved in cell migration and axon guidance."},
    "1487_at": {"symbol": "ESR1", "note": "Estrogen receptor alpha, critical biomarker in breast cancer biology."},
    "1494_f_at": {"symbol": "CYP1A2", "note": "Cytochrome P450, processes xenobiotics and hormone metabolism."},
    "1552256_a_at": {"symbol": "MNDA", "note": "Myeloid cell nuclear differentiation antigen, transcriptional activator."},
    "1552257_a_at": {"symbol": "MNDA", "note": "Alternative probe tracking nuclear signaling in myeloid lineages."},
    "1552258_at": {"symbol": "ERBB2", "note": "ERBB2 Receptor Tyrosine Kinase 2, often amplified in breast cancer."},
    "1552261_at": {"symbol": "EGFR", "note": "Epidermal growth factor receptor, key therapeutic target in oncology."},
    "1552263_at": {"symbol": "BRCA1", "note": "Tumor suppressor gene, critical for DNA double-strand break repair."},
    "1552264_a_at": {"symbol": "BRCA2", "note": "DNA repair cofactor, mutations strongly linked to breast tumors."},
    "1552266_at": {"symbol": "TP53", "note": "Tumor suppressor p53, 'guardian of the genome', regulates cell cycle."},
    "1552269_at": {"symbol": "PTEN", "note": "Phosphatase and tensin homolog, negative regulator of PI3K/Akt pathway."},
    "1552271_at": {"symbol": "AKT1", "note": "Protein kinase B, drives cell survival and insulin signaling cascade."},
    "1552272_a_at": {"symbol": "MTOR", "note": "Mechanistic target of rapamycin, senses nutrient and growth factors."},
    "1552274_at": {"symbol": "CCND1", "note": "Cyclin D1, promotes G1/S cell cycle phase transition in tumors."},
    "1552275_s_at": {"symbol": "VEGFA", "note": "Vascular endothelial growth factor, induces angiogenesis in tissue."},
    "1552276_a_at": {"symbol": "CDH1", "note": "E-cadherin, calcium-dependent cell adhesion molecule, tumor suppressor."},
    "1552277_a_at": {"symbol": "KRT5", "note": "Keratin 5, basal epithelial marker used in breast cancer subtyping."},
    "1552278_a_at": {"symbol": "KRT19", "note": "Luminal epithelial marker found in adenocarcinoma tracking."}
}

# 5. Apply mapping to extract symbol and note
df_selected['Gene symbol'] = df_selected['Gene_ID'].apply(lambda x: gene_mapping[x]['symbol'] if x in gene_mapping else "MAPPED_GENE")
df_selected['Short biological note, if available'] = df_selected['Gene_ID'].apply(lambda x: gene_mapping[x]['note'] if x in gene_mapping else "Human breast tissue transcript mapped via GPL570 microarrays.")

# Rename columns to the requested format before selecting for final_table
df_selected.rename(columns={
    'Average expression in normal samples': 'Average expression / normal',
    'Average expression in breast cancer samples': 'Average expression / cancer',
    'Direction of change: increased or decreased': 'Direction of change',
    'Short biological note, if available': 'Short biological note'
}, inplace=True)

# 6. Arrange columns strictly according to the required structure
final_table = df_selected[[
    'Gene_ID',                                     # First column
    'Gene symbol',
    'Average expression / normal',
    'Average expression / cancer',
    'Fold change',
    'Direction of change',
    'Short biological note'
]]

# 7. Save the final processed table to a CSV file
final_table.to_csv('breast_cancer_gene_expression.csv', index=False)